# CNN-BiLSTM baseline on DS-Base

Replicates the CNN-BiLSTM row for DS-Base in Table VI of the WAMM paper (arXiv:2512.23610). The run passes if accuracy and macro F1 are both within 5 percentage points of the paper.

The pipeline lives in `src/lib/baseline.py`. Progress is also appended to `results/progress.log`.

**Setup:** select the project's `.venv` as the kernel. The dataset downloads from R2 if it's missing (see `.env.example`).

In [ ]:
from lib import baseline

DATASET_NAME = "DS_Base.csv"
paper_accuracy, paper_macro_f1 = baseline.PAPER_CNN_BILSTM_RESULTS[DATASET_NAME]
print(f"Paper: accuracy {paper_accuracy}%, macro F1 {paper_macro_f1}%")

## Load and split

Stratified 80/20 split with a fixed seed. Only label counts are shown, because the requests contain attack payloads and cookies.

In [ ]:
import pandas as pd

request_texts, label_ids, class_names = baseline.load_dataset(DATASET_NAME)
train_texts, test_texts, train_labels, test_labels = baseline.split_train_test(request_texts, label_ids)
del request_texts, label_ids  # Free memory before training.

pd.DataFrame({
    "train": pd.Series(train_labels).value_counts().sort_index().to_numpy(),
    "test": pd.Series(test_labels).value_counts().sort_index().to_numpy(),
}, index=class_names)

## Train and evaluate

In [ ]:
cnn_bilstm_result = baseline.train_and_evaluate_cnn_bilstm(
    train_texts, test_texts, train_labels, test_labels, class_names
)

## Compare with the paper

In [ ]:
baseline.compare_with_paper(DATASET_NAME, cnn_bilstm_result)

## Per-class scores

In [ ]:
pd.DataFrame(cnn_bilstm_result["per_class_report"]).T.round(4)

## Confusion matrix

Rows are normalized, like Fig. 7 in the paper.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

figure, axes = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    test_labels, cnn_bilstm_result["predicted_labels"],
    labels=range(len(class_names)), display_labels=class_names,
    normalize="true", values_format=".2f", cmap="Blues", ax=axes, xticks_rotation=45,
)
axes.set_title(f"CNN-BiLSTM normalized confusion matrix ({DATASET_NAME})")
plt.tight_layout()
plt.show()

## Save the metrics

In [ ]:
print("Saved:", baseline.save_results(DATASET_NAME, cnn_bilstm_result, class_names))